# Feature Engineering - Supply Chain Optimization

Simple student-written code for feature engineering

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Use default colors
plt.style.use('default')
sns.set_palette('husl')

print('Libraries loaded successfully!')

## Load Data

In [ ]:
# Load all CSV files
customer = pd.read_csv('data/dim_customer.csv')
date = pd.read_csv('data/dim_date.csv')
facility = pd.read_csv('data/dim_facility.csv')
product = pd.read_csv('data/dim_product.csv')
supplier = pd.read_csv('data/dim_supplier.csv')

inventory = pd.read_csv('data/fact_inventory.csv')
procurement = pd.read_csv('data/fact_procurement.csv')
production = pd.read_csv('data/fact_production.csv')
sales = pd.read_csv('data/fact_sales.csv')
shipment = pd.read_csv('data/fact_shipment.csv')

# Show data shapes
print('Dimension Tables:')
print(f'Customer: {customer.shape}')
print(f'Date: {date.shape}')
print(f'Facility: {facility.shape}')
print(f'Product: {product.shape}')
print(f'Supplier: {supplier.shape}')
print()
print('Fact Tables:')
print(f'Inventory: {inventory.shape}')
print(f'Procurement: {procurement.shape}')
print(f'Production: {production.shape}')
print(f'Sales: {sales.shape}')
print(f'Shipment: {shipment.shape}')

## Join Tables

In [ ]:
# Merge sales with dimensions
sales_merged = sales.merge(customer, on='customer_id', how='left')
sales_merged = sales_merged.merge(product, on='product_id', how='left')
sales_merged = sales_merged.merge(date, left_on='date_key', right_on='date_key', how='left')

print('Sales merged shape:', sales_merged.shape)
print(sales_merged.head())

## Inventory Features

In [ ]:
# Make a copy to work with
inv = inventory.copy()

# Feature 1: Stock coverage ratio
inv['stock_coverage'] = inv['stock_level'] / (inv['safety_stock_level'] + 1)

# Feature 2: Reorder gap
inv['reorder_gap'] = inv['stock_level'] - inv['reorder_point']

# Feature 3: Stockout risk (1 if below reorder point, 0 otherwise)
inv['stockout_risk'] = (inv['stock_level'] < inv['reorder_point']).astype(int)

# Feature 4: Overstocked (1 if very high, 0 otherwise)
inv['is_overstocked'] = (inv['stock_level'] > inv['safety_stock_level'] * 2).astype(int)

print('Inventory features created')
print(inv[['stock_level', 'stock_coverage', 'reorder_gap', 'stockout_risk']].head())

## Procurement Features

In [ ]:
# Make a copy to work with
proc = procurement.copy()

# Feature 1: Cost per unit
proc['cost_per_unit'] = proc['total_cost'] / (proc['order_quantity'] + 1)

# Feature 2: Is urgent (lead time < 5 days)
proc['is_urgent'] = (proc['lead_time_days'] < 5).astype(int)

# Feature 3: Good quality (score > median)
quality_median = proc['quality_score'].median()
proc['good_quality'] = (proc['quality_score'] > quality_median).astype(int)

# Feature 4: Lead time category (Fast, Medium, Slow)
proc['lead_time_category'] = pd.cut(proc['lead_time_days'], 
                                      bins=3, 
                                      labels=['Fast', 'Medium', 'Slow'])

print('Procurement features created')
print(proc[['cost_per_unit', 'is_urgent', 'good_quality', 'lead_time_category']].head())

## Production Features

In [ ]:
# Make a copy to work with
prod = production.copy()

# Feature 1: Production efficiency
prod['efficiency'] = 100 - prod['defect_rate_pct']

# Feature 2: Good batch (defect rate < 5%)
prod['good_batch'] = (prod['defect_rate_pct'] < 5).astype(int)

# Feature 3: Has defects
prod['has_defects'] = (prod['defective_units'] > 0).astype(int)

# Feature 4: Waste cost
# Assume defective units lose 50% of value
prod['waste_cost'] = prod['defective_units'] * 0.5

print('Production features created')
print(prod[['efficiency', 'good_batch', 'has_defects', 'waste_cost']].head())

## Sales Features

In [ ]:
# Make a copy to work with
sal = sales.copy()

# Feature 1: Discount impact
sal['discount_impact'] = sal['gross_revenue'] - sal['net_revenue']

# Feature 2: Revenue per unit
sal['revenue_per_unit'] = sal['gross_revenue'] / (sal['quantity_sold'] + 1)

# Feature 3: High discount flag
sal['high_discount'] = (sal['discount_pct'] > 10).astype(int)

# Feature 4: High profit flag
profit_median = sal['profit'].median()
sal['high_profit'] = (sal['profit'] > profit_median).astype(int)

# Feature 5: Bulk order flag
qty_q75 = sal['quantity_sold'].quantile(0.75)
sal['is_bulk'] = (sal['quantity_sold'] > qty_q75).astype(int)

print('Sales features created')
print(sal[['discount_impact', 'revenue_per_unit', 'high_discount', 'high_profit']].head())

## Shipment Features

In [ ]:
# Make a copy to work with
ship = shipment.copy()

# Feature 1: Shipping cost per unit
ship['cost_per_unit'] = ship['shipping_cost'] / (ship['quantity'] + 1)

# Feature 2: Cost per kg
ship['cost_per_kg'] = ship['shipping_cost'] / (ship['total_weight_kg'] + 1)

# Feature 3: On time delivery
ship['on_time'] = (ship['status'] == 'Delivered').astype(int)

# Feature 4: Is delayed
ship['is_delayed'] = (ship['status'] == 'Delayed').astype(int)

print('Shipment features created')
print(ship[['cost_per_unit', 'cost_per_kg', 'on_time', 'is_delayed']].head())

## Categorical Encoding

In [ ]:
# Encode categorical columns in sales data
le_channel = LabelEncoder()
le_category = LabelEncoder()

sales_merged['channel_encoded'] = le_channel.fit_transform(sales_merged['channel_type'])
sales_merged['category_encoded'] = le_category.fit_transform(sales_merged['category'])

print('Categorical variables encoded')
print('Channel types:', sales_merged['channel_type'].unique())
print('Channel encoded:', sales_merged['channel_encoded'].unique())

## Supplier Aggregated Features

In [ ]:
# Group procurement by supplier
supplier_agg = proc.groupby('supplier_id').agg({
    'quality_score': 'mean',
    'lead_time_days': 'mean',
    'cost_per_unit': 'mean',
    'is_urgent': 'sum',
    'good_quality': 'mean'
}).reset_index()

supplier_agg.columns = ['supplier_id', 'avg_quality', 'avg_lead_time', 
                         'avg_cost', 'urgent_orders', 'reliability']

# Reliability index (0-100)
supplier_agg['reliability_index'] = supplier_agg['reliability'] * 100

print('Supplier aggregated features')
print(supplier_agg)

## Customer Aggregated Features

In [ ]:
# Group sales by customer
customer_agg = sales_merged.groupby('customer_id').agg({
    'quantity_sold': 'sum',
    'profit': 'sum',
    'net_revenue': 'sum',
    'discount_pct': 'mean',
    'profit_margin_pct': 'mean'
}).reset_index()

customer_agg.columns = ['customer_id', 'total_quantity', 'total_profit', 
                         'total_revenue', 'avg_discount', 'avg_margin']

print('Customer aggregated features')
print(customer_agg)

## Product Aggregated Features

In [ ]:
# Group sales by product
product_agg = sales_merged.groupby('product_id').agg({
    'quantity_sold': 'sum',
    'profit': 'sum',
    'net_revenue': 'sum',
    'profit_margin_pct': 'mean'
}).reset_index()

product_agg.columns = ['product_id', 'total_sold', 'total_profit', 
                        'total_revenue', 'avg_margin']

# Product value category
product_agg['value_category'] = pd.cut(product_agg['total_revenue'], 
                                         bins=3, 
                                         labels=['Low', 'Medium', 'High'])

print('Product aggregated features')
print(product_agg)

## Visualization - Feature Distributions

In [ ]:
# Create subplots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Feature Distributions', fontsize=16, fontweight='bold')

# Plot 1: Stock coverage
axes[0, 0].hist(inv['stock_coverage'], bins=30, color='steelblue', edgecolor='black')
axes[0, 0].set_title('Stock Coverage Ratio')
axes[0, 0].set_xlabel('Coverage')
axes[0, 0].set_ylabel('Frequency')

# Plot 2: Cost per unit
axes[0, 1].hist(proc['cost_per_unit'], bins=30, color='orange', edgecolor='black')
axes[0, 1].set_title('Procurement Cost per Unit')
axes[0, 1].set_xlabel('Cost')
axes[0, 1].set_ylabel('Frequency')

# Plot 3: Production efficiency
axes[0, 2].hist(prod['efficiency'], bins=30, color='green', edgecolor='black')
axes[0, 2].set_title('Production Efficiency')
axes[0, 2].set_xlabel('Efficiency %')
axes[0, 2].set_ylabel('Frequency')

# Plot 4: Profit per sale
axes[1, 0].hist(sal['profit'], bins=30, color='red', edgecolor='black')
axes[1, 0].set_title('Profit per Sale')
axes[1, 0].set_xlabel('Profit')
axes[1, 0].set_ylabel('Frequency')

# Plot 5: Shipping cost
axes[1, 1].hist(ship['cost_per_unit'], bins=30, color='purple', edgecolor='black')
axes[1, 1].set_title('Shipping Cost per Unit')
axes[1, 1].set_xlabel('Cost')
axes[1, 1].set_ylabel('Frequency')

# Plot 6: Customer revenue
axes[1, 2].hist(customer_agg['total_revenue'], bins=30, color='brown', edgecolor='black')
axes[1, 2].set_title('Customer Total Revenue')
axes[1, 2].set_xlabel('Revenue')
axes[1, 2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print('Visualization complete')

## Correlation Analysis

In [ ]:
# Select numeric columns from sales
numeric_cols = sales_merged.select_dtypes(include=[np.number]).columns
corr_matrix = sales_merged[numeric_cols].corr()

# Show correlation with profit
profit_corr = corr_matrix['profit'].sort_values(ascending=False)
print('\nTop Features Correlated with Profit:')
print(profit_corr.head(10))

# Visualize correlation matrix
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## Export Features

In [ ]:
# Create output directory
import os
os.makedirs('engineered_features', exist_ok=True)

# Save all engineered datasets
inv.to_csv('engineered_features/inventory_features.csv', index=False)
proc.to_csv('engineered_features/procurement_features.csv', index=False)
prod.to_csv('engineered_features/production_features.csv', index=False)
sal.to_csv('engineered_features/sales_features.csv', index=False)
ship.to_csv('engineered_features/shipment_features.csv', index=False)

supplier_agg.to_csv('engineered_features/supplier_profile.csv', index=False)
customer_agg.to_csv('engineered_features/customer_profile.csv', index=False)
product_agg.to_csv('engineered_features/product_profile.csv', index=False)

# Save merged sales data with all features
sales_merged.to_csv('engineered_features/sales_with_features.csv', index=False)

print('All engineered features exported!')
print('\nFiles saved:')
for file in os.listdir('engineered_features'):
    path = f'engineered_features/{file}'
    size = os.path.getsize(path) / 1024
    print(f'  {file:40s} - {size:>8.1f} KB')

## Summary

In [ ]:
print('='*70)
print('FEATURE ENGINEERING SUMMARY')
print('='*70)
print()
print('INVENTORY FEATURES:')
print('  - stock_coverage: Current stock / safety stock')
print('  - reorder_gap: Stock level - reorder point')
print('  - stockout_risk: Flag if stock < reorder point')
print('  - is_overstocked: Flag if stock > 2x safety stock')
print()
print('PROCUREMENT FEATURES:')
print('  - cost_per_unit: Total cost / order quantity')
print('  - is_urgent: Flag if lead time < 5 days')
print('  - good_quality: Flag if quality > median')
print('  - lead_time_category: Fast / Medium / Slow')
print()
print('PRODUCTION FEATURES:')
print('  - efficiency: 100 - defect_rate')
print('  - good_batch: Flag if defect rate < 5%')
print('  - has_defects: Flag if defective units > 0')
print('  - waste_cost: Defective units * 0.5')
print()
print('SALES FEATURES:')
print('  - discount_impact: Gross revenue - net revenue')
print('  - revenue_per_unit: Gross revenue / quantity')
print('  - high_discount: Flag if discount > 10%')
print('  - high_profit: Flag if profit > median')
print('  - is_bulk: Flag if quantity > 75th percentile')
print()
print('SHIPMENT FEATURES:')
print('  - cost_per_unit: Shipping cost / quantity')
print('  - cost_per_kg: Shipping cost / weight')
print('  - on_time: Flag if status = Delivered')
print('  - is_delayed: Flag if status = Delayed')
print()
print('AGGREGATED FEATURES:')
print('  - Supplier: avg quality, lead time, cost, reliability')
print('  - Customer: total quantity, profit, revenue, discounts')
print('  - Product: total sold, profit, revenue, margin, value category')
print()
print('='*70)